# Experiment — Traces

A trace records what an application received and returned. It can help locate an unsupported claim by keeping the instructions, request, supplied information and response together. A trace does not reveal hidden model reasoning or prove the internal cause of an error.

You will inspect a fictional shop-support interaction that exactly matches the book illustration. The supplied record says a return was received and its refund is pending, while the offline response incorrectly says it was paid. Predict the contradiction, run the recorded interaction and locate the evidence. No shop system or tool is contacted. The unchanged helper is included; offline mode makes no API calls, and optional local live mode makes one request and may produce a different, correct response.


## 1. Prepare and predict

Run from the top in a fresh kernel. The small source record is embedded below, so the notebook works without earlier state or downloaded data. The accompanying JSON files preserve the same inputs and reference for inspection.

Before generating a response, write what the record supports. Does return received establish that a refund was paid? Identify the field that answers the customer's question.


In [ ]:
lookup_record = {'order_number': 'O1001', 'return_status': 'received', 'refund_status': 'pending'}
question = "Is my refund paid?"
print(lookup_record)


## 2. Reuse the request helper

The following is the unchanged get_completion(system_prompt, user_prompt) adapter used earlier. Its record of requests contains observable messages and outputs, not hidden reasoning. Keep offline mode in browser Python. To use OpenAI locally, install the companion requirements and keep one private .env at the extracted package root beside COMPANION_VERSION (or beside this notebook when standalone). Supply your own OPENAI_API_KEY and OPENAI_MODEL. Never put a key in cells or outputs.


In [ ]:
# Shared offline/live request adapter. No API key is stored in this notebook.
# Code origin: get_completion(system_prompt, user_prompt) was introduced in Section 2.3.
# Notebook extension used from Section 3.2: supplied/live mode and interaction records.
# Evaluation notebooks reuse this helper without changing its two-argument interface.
import json
import os
import sys

MODE = "offline"
MODEL = ""
_client = None
request_records = []
OFFLINE_SAMPLES = {}

# Match supplied examples only when both messages are unchanged.
def _request_key(system_prompt, user_prompt):
    return json.dumps([system_prompt.strip(), user_prompt.strip()], ensure_ascii=False)

# Register a supplied teaching response; this does not contact OpenAI.
def register_sample(system_prompt, user_prompt, response):
    OFFLINE_SAMPLES[_request_key(system_prompt, user_prompt)] = response

# Locate one package-root configuration, while retaining standalone notebook use.
def _configuration_file():
    from pathlib import Path
    bases = [Path.cwd()]
    if "__file__" in globals():
        bases.insert(0, Path(__file__).resolve().parent)
    for base in bases:
        for folder in [base, *base.parents]:
            if (folder / "COMPANION_VERSION").is_file():
                return folder / ".env"
    # Standalone notebooks keep their configuration beside the notebook.
    return bases[0] / ".env"

# Load the private configuration and prepare an OpenAI client when live mode is chosen.
def configure_live(model=""):
    global MODE, MODEL, _client
    if sys.platform in {"emscripten", "wasi"}:
        raise RuntimeError("The OpenAI SDK needs a standard Python kernel. Keep offline mode in browser Python.")
    try:
        from openai import OpenAI
        from dotenv import load_dotenv
    except ImportError:
        raise RuntimeError("Install openai and python-dotenv in your Python kernel, then rerun setup.") from None
    # Read the key and model from the local file without printing the key.
    load_dotenv(_configuration_file())
    selected_model = model.strip() or os.getenv("OPENAI_MODEL", "").strip()
    if not selected_model or selected_model.startswith("YOUR_"):
        raise ValueError("Set OPENAI_MODEL in your .env file before choosing live mode.")
    key = os.getenv("OPENAI_API_KEY", "").strip()
    if not key or key.startswith("YOUR_"):
        raise ValueError("Copy your own OpenAI key into the .env file before choosing live mode.")
    _client = OpenAI(timeout=45.0, max_retries=0)
    MODEL = selected_model
    MODE = "live"
    print("OpenAI configured from .env. No key is printed.")


def get_completion(system_prompt, user_prompt):
    if MODE == "live":
        if _client is None:
            raise RuntimeError("Run configure_live(MODEL) in a private Python kernel first.")
        try:
            # Send role instructions separately from the user task and data.
            response = _client.responses.create(model=MODEL, instructions=system_prompt, input=user_prompt, max_output_tokens=1000, store=False)
        except Exception:
            raise RuntimeError("API request failed. Check your model, account access and connection; no key has been printed.") from None
        text = response.output_text
        print("[Live OpenAI response]")
    else:
        # Offline mode returns stored text, not a newly generated model answer.
        text = OFFLINE_SAMPLES.get(_request_key(system_prompt, user_prompt))
        if text is None:
            print("[No saved response for these edited instructions]")
            return "No model was called. Use OpenAI mode in a standard Python kernel, or label a response you write yourself."
        print("[Supplied teaching response — no API call]")
    # Keep messages and responses for evaluation; never record the API key.
    request_records.append({"mode": MODE, "model": MODEL if MODE == "live" else "supplied-example", "system_prompt": system_prompt, "user_prompt": user_prompt, "response": text})
    return text

# Release the API client after finishing the experiment.
def clear_live():
    global _client, MODE
    if _client is not None:
        _client.close()
    _client = None
    MODE = "offline"
    print("Live client cleared from this kernel.")

In [ ]:
MODE = "offline"
if MODE == "live":
    configure_live()


## 3. Run the interaction

The lookup record is data supplied in the user request, not a tool call. The offline reply below is a deliberately incorrect teaching response for these unchanged inputs. It is not a recorded model output. A live model may correctly report pending; inspect its actual response rather than expecting the same error.


In [ ]:
system_prompt = 'You are a shop support assistant. Explain the supplied record only. Do not claim a refund is paid when its recorded status is pending.'
user_prompt = question + "\nLOOKUP RECORD:\n" + json.dumps(lookup_record, indent=2)
register_sample('You are a shop support assistant. Explain the supplied record only. Do not claim a refund is paid when its recorded status is pending.', 'Is my refund paid?\nLOOKUP RECORD:\n{\n  "order_number": "O1001",\n  "return_status": "received",\n  "refund_status": "pending"\n}', "Refund paid.")
response = get_completion(system_prompt, user_prompt)


## 4. Inspect the trace

Keep the instruction, actual input, source record, response and execution mode together. Find where the paid claim first appears. Is it in the supplied record or only in the response? The trace shows the discrepancy, but does not establish why the model made it.


In [ ]:
trace = {"system_prompt": system_prompt, "user_request": user_prompt,
         "supplied_record": lookup_record, "response": response, "mode": MODE}
print(json.dumps(trace, indent=2))
print("REFERENCE REFUND STATUS:", lookup_record["refund_status"])
print("REPORTED RESPONSE:", response)


## 5. Change and explain

Change the recorded refund status from pending to paid while keeping the rest of the record and the instructions fixed. Predict what should change in the response. An edited request has no new offline model answer; describe a proposed experiment or use optional local live mode for a fresh response. Do not present a manually supplied reply as measured model behaviour.

Explain why return received and refund paid are different claims, what evidence supports the expected answer, and which trace fields let you locate the contradiction. Save the trace and your short explanation. No automatic retry or feedback loop is introduced here.

**Check your work:** the original source supports pending. In the offline example the unsupported paid claim appears in the response. For a live response, compare the actual wording with the same record; it may already be correct.


In [ ]:
clear_live()